Exploratory notebook: outputs and row-level examples are intentionally omitted from Git. This notebook predates the audited Purchase-only 12-month label; use docs/data-contract.md and scripts/label/label_rules.py for the current study. Run only with local Freddie Mac data; it is not a synthetic quick-start.

# EDA bảng `perf` (lịch sử trả nợ), chạy **từng quarter**, DuckDB

`perf` có ~533 triệu dòng nên notebook này **xử lý lần lượt từng quarter** (mỗi quarter = 1 file Parquet) và lưu kết quả đã tổng hợp vào `eda_perf.duckdb`.
- **Chạy lại được:** quarter nào xong rồi sẽ được bỏ qua; bị ngắt giữa chừng thì chạy lại là tiếp tục.
- **RAM thấp:** mỗi lúc chỉ xử lý 1 file.
- Phần `orig` nằm ở notebook riêng `eda_orig.ipynb`. Notebook này chỉ đọc `orig` để kiểm tra khoản vay mồ côi và để tính tỷ lệ vỡ nợ.

**Cấu trúc:**
- **Phần A:** Setup và cấu hình
- **Phần B:** Hiểu cấu trúc `perf` (Bước 1, chạy trên 1 quarter mẫu)
- **Phần C:** Chạy từng quarter, lưu kết quả (Bước 2 và 3 được tính ở đây)
- **Phần D:** Đọc kết quả: chất lượng, đơn biến, xu hướng theo quarter (Bước 2, 3)
- **Phần E:** Vỡ nợ, kết hợp với `orig` (Bước 4)
- **Phần F:** Insight (Bước 5)

**Cách chạy:** A → B → C (thử 1 quarter trước, rồi chạy tất cả) → D → E → F.

## Phần A. Setup

In [ ]:
import duckdb, glob, os, time
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

# ====== CẤU HÌNH ======
from pathlib import Path
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/pilot_baseline.yaml").is_file())
DATA_DIR = str(PROJECT_ROOT / "parquet")
MEMORY_LIMIT = "6GB"
THREADS      = 4
DB_FILE      = "eda_perf.duckdb"

LOAN_ID     = "loan_id"
PERF_PERIOD = "monthly_reporting_period"
DELINQ      = "current_loan_delinquency_status"
LOAN_AGE    = "loan_age"                # tuổi khoản vay (tháng)
OBS_MONTHS  = 24                        # cửa sổ quan sát vỡ nợ (tháng đầu của khoản vay)

SAMPLE_QUARTER   = "2023Q1"             # quarter mẫu cho Phần B
QUARTERS_TO_RUN  = None           # Phần C. Thử trước 1 quarter. Đặt None = chạy tất cả quarter chưa xong
FORCE_RERUN      = True                # True = tính lại cả quarter đã xong

# Quy tắc giá trị không hợp lệ cho perf (điều kiện SQL mô tả giá trị SAI)
RULES_PERF = {
    "du_no_am":              "current_upb < 0",
    "lai_suat_hien_tai_la":  "current_interest_rate <= 0 OR current_interest_rate > 20",
}

In [ ]:
con = duckdb.connect(DB_FILE)
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'; SET threads={THREADS}; SET temp_directory='./duckdb_tmp';")

def run(sql, params=None):
    return con.execute(sql, params).df() if params else con.execute(sql).df()

def table_exists(t):
    return con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?", [t]).fetchone()[0] > 0

QUARTERS = sorted(os.path.basename(p).split("=")[1] for p in glob.glob(f"{DATA_DIR}/perf/quarter=*"))
print(f"Tìm thấy {len(QUARTERS)} quarter trong perf: {QUARTERS[0]} → {QUARTERS[-1]}")

def perf_path(qtr): return f"{DATA_DIR}/perf/quarter={qtr}/data.parquet"
def orig_path(qtr): return f"{DATA_DIR}/orig/quarter={qtr}/data.parquet"

def use_quarter(qtr):
    """Tạo view `cur` trỏ tới đúng 1 file perf của quarter."""
    con.execute(f"CREATE OR REPLACE VIEW cur AS SELECT * FROM read_parquet('{perf_path(qtr)}')")

## Phần B. Bước 1: Hiểu cấu trúc `perf` (trên 1 quarter mẫu)

In [ ]:
use_quarter(SAMPLE_QUARTER)
desc_perf = run("DESCRIBE cur")
display(desc_perf[["column_name", "column_type", "null"]])
display(run("SELECT * FROM cur LIMIT 5"))
print("Số dòng quarter mẫu:", f"{con.execute('SELECT COUNT(*) FROM cur').fetchone()[0]:,}")

### Schema có đồng nhất giữa các quarter không

In [ ]:
r = run(f"""
    SELECT name, COUNT(DISTINCT type) AS so_kieu, list(DISTINCT type) AS cac_kieu, COUNT(DISTINCT file_name) AS so_file
    FROM parquet_schema('{DATA_DIR}/perf/*/data.parquet') WHERE name <> 'schema'
    GROUP BY name ORDER BY so_kieu DESC, name""")
bad = r[(r.so_kieu > 1) | (r.so_file < r.so_file.max())]
print("Số cột lệch kiểu / thiếu ở một số quarter:", len(bad)); display(bad)

### SUMMARIZE (quarter mẫu)

In [ ]:
display(run("SUMMARIZE cur"))

### Cột số dùng để profile (loại cột ngày / kỳ báo cáo)

In [ ]:
NUM_TYPES = ("TINYINT","SMALLINT","INTEGER","BIGINT","HUGEINT","UTINYINT","USMALLINT",
             "UINTEGER","UBIGINT","FLOAT","DOUBLE","DECIMAL")
perf_num = [c for c, t in zip(desc_perf.column_name, desc_perf.column_type)
            if t.upper().startswith(NUM_TYPES) and not any(k in c for k in ("date", "period"))]
perf_cat = [c for c, t in zip(desc_perf.column_name, desc_perf.column_type) if t.upper() == "VARCHAR" and c != "quarter"]
for col in [LOAN_ID, PERF_PERIOD, DELINQ, LOAN_AGE]:
    if col not in desc_perf.column_name.values:
        print(f"⚠️ Không thấy cột '{col}' trong perf, hãy sửa cấu hình ở Phần A")
print("Cột số:", perf_num); print("Cột phân loại:", perf_cat)
display(run(f'SELECT "{DELINQ}" AS gia_tri, COUNT(*) AS n FROM cur GROUP BY 1 ORDER BY n DESC LIMIT 20'))

**Data dictionary:**

| Cột | Kiểu | Ý nghĩa | Ghi chú |
|---|---|---|---|
| | | | |

## Phần C. Chạy từng quarter

Với mỗi quarter, hàm `process_quarter` tính và lưu vào `eda_perf.duckdb`:

| Bảng | Nội dung | Bước |
|---|---|---|
| `q_overview` | số dòng, số khoản vay, kỳ min/max, **trùng khóa**, **mồ côi** so với `orig` cùng quarter | 1, 2 |
| `q_nulls` | số NULL từng cột | 2 |
| `q_rules` | số dòng vi phạm từng quy tắc | 2 |
| `q_numstats` | n, mean, std, min, Q1, median, Q3, max từng cột số (quantile gần đúng) | 3 |
| `q_delinq` | phân phối trạng thái trễ hạn | 3 |
| `loan_perf_agg` | mỗi khoản vay 1 dòng: số tháng, tuổi tối đa, từng vỡ nợ, vỡ nợ trong `OBS_MONTHS` tháng đầu | 4 |

`q_overview` được ghi **cuối cùng** và dùng làm dấu hiệu "quarter này đã xong".

In [ ]:
def save(table, qtr, df):
    """Ghi df vào bảng (xóa dữ liệu cũ của quarter đó trước)."""
    df = df.copy(); df.insert(0, "quarter", qtr)
    con.register("df_tmp", df)
    if table_exists(table):
        con.execute(f"DELETE FROM {table} WHERE quarter = ?", [qtr])
        con.execute(f"INSERT INTO {table} SELECT * FROM df_tmp")
    else:
        con.execute(f"CREATE TABLE {table} AS SELECT * FROM df_tmp")
    con.unregister("df_tmp")

DEFAULT_COND = f"""(TRY_CAST("{DELINQ}" AS INTEGER) >= 3 OR "{DELINQ}" = 'RA')"""   # vỡ nợ: trễ >= 90 ngày hoặc REO

def process_quarter(qtr):
    t0 = time.time()
    use_quarter(qtr)
    desc = run("DESCRIBE cur")
    cols = desc.column_name.tolist()
    num = [c for c, t in zip(desc.column_name, desc.column_type)
           if t.upper().startswith(NUM_TYPES) and not any(k in c for k in ("date", "period"))]

    # --- tổng quan + trùng khóa
    n_rows, n_loans, p_min, p_max = con.execute(
        f'SELECT COUNT(*), COUNT(DISTINCT "{LOAN_ID}"), MIN("{PERF_PERIOD}"), MAX("{PERF_PERIOD}") FROM cur').fetchone()
    n_dup = con.execute(f"""SELECT COUNT(*) FROM (SELECT 1 FROM cur GROUP BY "{LOAN_ID}", "{PERF_PERIOD}" HAVING COUNT(*) > 1)""").fetchone()[0]

    # --- mồ côi so với orig cùng quarter
    n_orphan = n_orig_no_perf = None
    if os.path.exists(orig_path(qtr)):
        n_orphan = con.execute(f"""SELECT COUNT(*) FROM (SELECT DISTINCT "{LOAN_ID}" FROM cur) p
                                   ANTI JOIN read_parquet('{orig_path(qtr)}') o USING ("{LOAN_ID}")""").fetchone()[0]
        n_orig_no_perf = con.execute(f"""SELECT COUNT(*) FROM read_parquet('{orig_path(qtr)}') o
                                         ANTI JOIN (SELECT DISTINCT "{LOAN_ID}" FROM cur) p USING ("{LOAN_ID}")""").fetchone()[0]

    # --- NULL từng cột (1 lần quét)
    sel = ", ".join(f'COUNT(*) - COUNT("{c}") AS "{c}"' for c in cols)
    r = run(f"SELECT {sel} FROM cur").iloc[0]
    nulls = pd.DataFrame({"col": r.index, "n_null": r.values.astype("int64"), "total": n_rows})
    save("q_nulls", qtr, nulls)

    # --- quy tắc không hợp lệ
    rules = []
    for name, cond in RULES_PERF.items():
        try: rules.append((name, con.execute(f"SELECT COUNT(*) FILTER (WHERE {cond}) FROM cur").fetchone()[0]))
        except Exception as e: rules.append((name, None)); print(f"   ⚠️ quy tắc {name}: {str(e)[:80]}")
    if rules: save("q_rules", qtr, pd.DataFrame(rules, columns=["rule", "n_bad"]))

    # --- thống kê cột số (quantile gần đúng để nhanh)
    if num:
        parts = []
        for c in num:
            parts += [f'COUNT("{c}") AS "{c}__n"', f'AVG("{c}") AS "{c}__mean"', f'STDDEV("{c}") AS "{c}__std"',
                      f'MIN("{c}") AS "{c}__min"', f'approx_quantile("{c}", 0.25) AS "{c}__q1"',
                      f'approx_quantile("{c}", 0.5) AS "{c}__median"', f'approx_quantile("{c}", 0.75) AS "{c}__q3"',
                      f'MAX("{c}") AS "{c}__max"']
        r = run("SELECT " + ", ".join(parts) + " FROM cur").iloc[0]
        stats = pd.DataFrame([{"col": c, **{k: float(r[f"{c}__{k}"]) if pd.notna(r[f"{c}__{k}"]) else None
                                            for k in ["n", "mean", "std", "min", "q1", "median", "q3", "max"]}} for c in num])
        save("q_numstats", qtr, stats)

    # --- phân phối trạng thái trễ hạn
    save("q_delinq", qtr, run(f'SELECT "{DELINQ}"::VARCHAR AS status, COUNT(*) AS n FROM cur GROUP BY 1'))

    # --- tổng hợp mức khoản vay (cho phân tích vỡ nợ ở Phần E)
    con.execute("""CREATE TABLE IF NOT EXISTS loan_perf_agg (quarter VARCHAR, loan_id VARCHAR, so_thang BIGINT,
                   tuoi_toi_da BIGINT, tung_vo_no INTEGER, vo_no_obs INTEGER)""")
    con.execute("DELETE FROM loan_perf_agg WHERE quarter = ?", [qtr])
    con.execute(f"""
        INSERT INTO loan_perf_agg
        SELECT '{qtr}', "{LOAN_ID}", COUNT(*), MAX("{LOAN_AGE}")::BIGINT,
               MAX(CASE WHEN {DEFAULT_COND} THEN 1 ELSE 0 END)::INTEGER,
               MAX(CASE WHEN {DEFAULT_COND} AND "{LOAN_AGE}" <= {OBS_MONTHS} THEN 1 ELSE 0 END)::INTEGER
        FROM cur GROUP BY 1, 2""")

    # --- đánh dấu hoàn tất (ghi cuối cùng)
    secs = round(time.time() - t0, 1)
    save("q_overview", qtr, pd.DataFrame([{"n_rows": n_rows, "n_loans": n_loans, "period_min": p_min, "period_max": p_max,
                                           "n_dup_keys": n_dup, "n_orphan_perf": n_orphan,
                                           "n_orig_no_perf": n_orig_no_perf, "seconds": secs}]))
    return n_rows, secs

In [ ]:
done = set(run("SELECT quarter FROM q_overview").quarter) if table_exists("q_overview") else set()
todo = [x for x in (QUARTERS if QUARTERS_TO_RUN is None else QUARTERS_TO_RUN) if FORCE_RERUN or x not in done]
print(f"Đã xong: {len(done)} quarter | Sẽ chạy: {len(todo)} quarter -> {todo}")

for i, qtr in enumerate(todo, 1):
    try:
        n, secs = process_quarter(qtr)
        print(f"[{i}/{len(todo)}] {qtr}: {n:,} dòng, {secs}s")
    except Exception as e:
        print(f"[{i}/{len(todo)}] {qtr}: LỖI -> {str(e)[:200]} (quarter này chưa được đánh dấu xong, sẽ chạy lại lần sau)")
print("Hoàn tất lượt chạy.")

## Phần D. Đọc kết quả (Bước 2 và 3)

### D.1 Tổng quan theo quarter, trùng khóa và khoản vay mồ côi

In [ ]:
ov = run("SELECT * FROM q_overview ORDER BY quarter")
display(ov)
print(f"Đã xử lý {len(ov)}/{len(QUARTERS)} quarter | tổng {ov.n_rows.sum():,.0f} dòng perf")
if len(ov) > 1:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ov.plot.bar(x="quarter", y="n_rows", ax=ax[0], legend=False, title="Số dòng perf theo quarter")
    ov.plot.bar(x="quarter", y="n_loans", ax=ax[1], legend=False, title="Số khoản vay có trong perf theo quarter")
    plt.tight_layout(); plt.show()
bad = ov[(ov.n_dup_keys > 0) | (ov.n_orphan_perf.fillna(0) > 0) | (ov.n_orig_no_perf.fillna(0) > 0)]
print("Quarter có trùng khóa / mồ côi:", bad.quarter.tolist() if len(bad) else "không có")

### D.2 Giá trị thiếu theo quarter (% NULL)

In [ ]:
nl = run("SELECT quarter, col, 100.0 * n_null / total AS pct_null FROM q_nulls")
pv = nl.pivot(index="col", columns="quarter", values="pct_null")
always100 = pv.index[(pv.min(axis=1) >= 99.99)]
partial = pv.loc[(pv.max(axis=1) > 0) & ~pv.index.isin(always100)]
print(f"{len(always100)} cột NULL 100% ở mọi quarter (thường là cột chỉ có dữ liệu khi thanh lý / có sự kiện đặc biệt):")
print(list(always100))
print("\nCột NULL một phần (% theo quarter):")
display(partial.round(2))

### D.3 Giá trị không hợp lệ theo quarter

In [ ]:
if table_exists("q_rules"):
    display(run("SELECT quarter, rule, n_bad FROM q_rules").pivot(index="quarter", columns="rule", values="n_bad"))

### D.4 Đơn biến: thống kê cột số theo quarter

In [ ]:
ns = run("SELECT * FROM q_numstats ORDER BY quarter, col")
cols_show = [c for c in ["current_upb", "current_interest_rate", LOAN_AGE] if c in set(ns.col)] or ns.col.unique()[:3].tolist()
for c in cols_show:
    print(f"--- {c} ---"); display(ns[ns.col == c].drop(columns="col").set_index("quarter").round(2))
if ns.quarter.nunique() > 1:
    fig, axes = plt.subplots(1, len(cols_show), figsize=(5*len(cols_show), 3.5))
    for ax, c in zip(np.atleast_1d(axes), cols_show):
        d = ns[ns.col == c].set_index("quarter")
        d[["mean", "median"]].plot(ax=ax, marker="o", title=f"{c}: mean / median theo quarter"); ax.tick_params(axis="x", rotation=60)
    plt.tight_layout(); plt.show()
print("Lưu ý: mỗi quarter có cơ cấu 'tuổi khoản vay' khác nhau nên không so sánh trực tiếp các giá trị trung bình giữa quarter như so sánh cùng độ tuổi.")

### D.5 Đơn biến: phân phối trạng thái trễ hạn

In [ ]:
dl = run("SELECT quarter, status, n FROM q_delinq")
tot = dl.groupby("status").n.sum().sort_values(ascending=False)
display(pd.DataFrame({"n": tot, "pct": (tot / tot.sum() * 100).round(3)}).head(20))
pv = dl.pivot(index="quarter", columns="status", values="n").fillna(0)
share = pv.div(pv.sum(axis=1), axis=0) * 100
top = tot.index[:6].tolist()
if len(share) > 1:
    share[[c for c in top if c != "00"]].plot(figsize=(11, 4), marker="o", title="% dòng theo trạng thái trễ hạn (trừ '00') theo quarter")
    plt.ylabel("%"); plt.xticks(rotation=60); plt.show()

## Phần E. Bước 4: Vỡ nợ, kết hợp `perf` (`loan_perf_agg`) với `orig`

**Định nghĩa vỡ nợ** (sửa `DEFAULT_COND` ở Phần C nếu đề bài khác): trễ từ **90 ngày** (`delinquency >= 3`) hoặc `'RA'` (REO), **trong `OBS_MONTHS` tháng đầu**.
**Right-censoring:** chỉ so sánh các quarter khởi tạo đủ `OBS_MONTHS` tháng quan sát so với tháng dữ liệu mới nhất.

In [ ]:
# View orig (đã đổi mã đặc biệt thành NULL) để join
SPECIAL_CODES = {"credit_score": 9999, "vantage_score_4": 9999, "dti": 999, "ltv": 999, "cltv": 999, "mi_percentage": 999}
_oc = set(run(f"DESCRIBE SELECT * FROM read_parquet('{DATA_DIR}/orig/*/data.parquet', union_by_name=true)").column_name)
SPECIAL_CODES = {c: v for c, v in SPECIAL_CODES.items() if c in _oc}
repl = ", ".join(f'NULLIF("{c}", {v}) AS "{c}"' for c, v in SPECIAL_CODES.items())
con.execute(f"""CREATE OR REPLACE VIEW orig_c AS
    SELECT * REPLACE ({repl}) FROM read_parquet('{DATA_DIR}/orig/*/data.parquet', hive_partitioning=true, union_by_name=true)""")

done_q = run("SELECT quarter, period_max FROM q_overview ORDER BY quarter")
last_ym = int(done_q.period_max.max())
def eligible(qq, obs):
    y, k = int(qq[:4]), int(qq[-1])
    return y*12 + 3*k + obs <= (last_ym // 100)*12 + last_ym % 100
ELIGIBLE = [x for x in done_q.quarter if eligible(x, OBS_MONTHS)]
elig_sql = ", ".join(f"'{x}'" for x in ELIGIBLE) or "''"
print(f"Tháng dữ liệu mới nhất: {last_ym} | quarter đủ {OBS_MONTHS} tháng quan sát: {ELIGIBLE[0] if ELIGIBLE else None} → {ELIGIBLE[-1] if ELIGIBLE else None}")
print(f"Số khoản vay trong loan_perf_agg: {con.execute('SELECT COUNT(*) FROM loan_perf_agg').fetchone()[0]:,}")

### E.1 Tỷ lệ vỡ nợ theo quarter khởi tạo

In [ ]:
r = run(f"""
    SELECT p.quarter, COUNT(*) AS n, ROUND(100*AVG(p.vo_no_obs), 3) AS pct_vo_no_obs,
           ROUND(100*AVG(p.tung_vo_no), 3) AS pct_tung_vo_no_toan_ky, ROUND(AVG(p.tuoi_toi_da), 1) AS tuoi_tb
    FROM loan_perf_agg p GROUP BY 1 ORDER BY 1""")
r["du_quan_sat"] = r.quarter.isin(ELIGIBLE)
display(r)
if len(r) > 1:
    ax = r.plot(x="quarter", y=["pct_vo_no_obs", "pct_tung_vo_no_toan_ky"], marker="o",
                title="% vỡ nợ theo quarter khởi tạo (quarter cuối chưa đủ quan sát → thấp giả tạo)")
    ax.set_ylabel("%"); plt.xticks(rotation=60); plt.tight_layout(); plt.show()

### E.2 Vỡ nợ theo điểm tín dụng, LTV, DTI, mục đích vay (chỉ quarter đủ quan sát)

In [ ]:
BIN = 50
r = run(f"""
    SELECT FLOOR(o.credit_score/{BIN})*{BIN} AS nhom_fico, COUNT(*) AS n, ROUND(100*AVG(p.vo_no_obs), 3) AS pct_vo_no
    FROM loan_perf_agg p JOIN orig_c o USING (loan_id)
    WHERE p.quarter IN ({elig_sql}) AND o.credit_score IS NOT NULL GROUP BY 1 ORDER BY 1""")
display(r)
if len(r):
    ax = r[r.n >= 100].plot(x="nhom_fico", y="pct_vo_no", marker="o", legend=False,
                            title=f"% vỡ nợ trong {OBS_MONTHS} tháng đầu theo điểm tín dụng (nhóm ≥100 khoản vay)")
    ax.set_ylabel("%"); plt.show()

In [ ]:
for col, width in [("ltv", 10), ("dti", 10)]:
    display(run(f"""
        SELECT FLOOR(o.{col}/{width})*{width} AS nhom_{col}, COUNT(*) AS n, ROUND(100*AVG(p.vo_no_obs), 3) AS pct_vo_no
        FROM loan_perf_agg p JOIN orig_c o USING (loan_id)
        WHERE p.quarter IN ({elig_sql}) AND o.{col} IS NOT NULL GROUP BY 1 ORDER BY 1"""))
display(run(f"""
    SELECT o.loan_purpose, COUNT(*) AS n, ROUND(100*AVG(p.vo_no_obs), 3) AS pct_vo_no
    FROM loan_perf_agg p JOIN orig_c o USING (loan_id) WHERE p.quarter IN ({elig_sql}) GROUP BY 1 ORDER BY n DESC"""))

## Phần F. Bước 5: Tổng hợp insight (perf)

### Phát hiện chính
1. ...
2. ...
3. ...

### Hạn chế
- **Right-censoring:** quarter mới chưa đủ lịch sử, chỉ so sánh các quarter "đủ quan sát"
- Cột thiếu 100% là thiếu có cấu trúc (chỉ có dữ liệu khi có sự kiện thanh lý), không phải lỗi dữ liệu
- Quantile trong `q_numstats` là gần đúng
- Định nghĩa vỡ nợ: ...

### Việc cần làm tiếp
- [ ] Kiểm tra quarter có trùng khóa / mồ côi (D.1)
- [ ] Phân tích sống sót (survival analysis) nếu cần xử lý right-censoring chặt hơn
- [ ] Kiểm định thống kê (t-test, chi-square)